# Exercise 1 (Instructor Solution) — Data Ingestion, Validation & Row Quarantine

Complete reference solution for Exercise 1, including the Bonus Stretch Task (`failure_reason` annotation).


In [ ]:
from pathlib import Path
import pandas as pd

BATCH_PATH = Path("data/incoming_batch_exercise.csv")
raw_batch = pd.read_csv(BATCH_PATH)

print(f"Loaded raw batch: {raw_batch.shape}")
print("Unique customer_ids:", raw_batch["customer_id"].nunique())


In [ ]:
def clean_and_deduplicate(df: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy of df with normalized text and unique customer_ids."""
    cleaned = df.copy()
    cleaned["plan_tier"] = cleaned["plan_tier"].str.strip().str.lower()
    cleaned["country"] = cleaned["country"].str.strip().str.upper()
    cleaned = cleaned.drop_duplicates(subset=["customer_id"], keep="first").reset_index(drop=True)
    return cleaned


In [ ]:
ALLOWED_PLANS = {"basic", "standard", "premium"}
ALLOWED_COUNTRIES = {"US", "IN", "UK", "CA", "DE"}


def split_valid_and_quarantine(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split df into (valid_df, quarantine_df) based on StreamPulse data contract."""
    rule_age = df["age"].between(18, 100)
    rule_tenure = df["tenure_months"] >= 0
    rule_charges = df["monthly_charges"].between(0, 100)
    rule_plan = df["plan_tier"].isin(ALLOWED_PLANS)
    rule_country = df["country"].isin(ALLOWED_COUNTRIES)

    is_valid = rule_age & rule_tenure & rule_charges & rule_plan & rule_country

    valid_df = df[is_valid].reset_index(drop=True)
    quarantine_df = df[~is_valid].copy().reset_index(drop=True)

    # Bonus: annotate exact failure reasons on each quarantined row
    def explain_row(row) -> str:
        reasons = []
        if not (18 <= row["age"] <= 100):
            reasons.append("age_out_of_range")
        if row["tenure_months"] < 0:
            reasons.append("negative_tenure")
        if not (0 <= row["monthly_charges"] <= 100):
            reasons.append("charges_out_of_range")
        if row["plan_tier"] not in ALLOWED_PLANS:
            reasons.append("unknown_plan_tier")
        if row["country"] not in ALLOWED_COUNTRIES:
            reasons.append("unknown_country")
        return "; ".join(reasons)

    if len(quarantine_df) > 0:
        quarantine_df["failure_reason"] = quarantine_df.apply(explain_row, axis=1)

    return valid_df, quarantine_df


In [ ]:
# Run self-check tests
cleaned_df = clean_and_deduplicate(raw_batch)
valid_df, quarantine_df = split_valid_and_quarantine(cleaned_df)

print(f"Cleaned rows    : {len(cleaned_df)}")
print(f"Valid rows      : {len(valid_df)}")
print(f"Quarantined rows: {len(quarantine_df)}")

assert cleaned_df["customer_id"].is_unique
assert len(cleaned_df) == 120
assert len(quarantine_df) == 4
assert len(valid_df) == 116
assert set(valid_df["plan_tier"].unique()).issubset(ALLOWED_PLANS)
assert set(valid_df["country"].unique()).issubset(ALLOWED_COUNTRIES)
assert valid_df["age"].between(18, 100).all()
assert (valid_df["tenure_months"] >= 0).all()
assert valid_df["monthly_charges"].between(0, 100).all()

# Save quarantined batch for data engineering follow-up
out_path = Path("artifacts/quarantined_batch.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
quarantine_df.to_csv(out_path, index=False)

print(f"\n[ALL TESTS PASSED] Saved quarantined rows to {out_path}")
quarantine_df[["customer_id", "age", "plan_tier", "tenure_months", "monthly_charges", "failure_reason"]]
